In [8]:
from braindecode.models import EEGConformer
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm
import sklearn 

class Dataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

def train_one_epoch(dataloader: DataLoader,model,loss_fn,optimizer,scheduler,epoch: int,device,print_batch_stats=True):  
    model.train()  # Set the model to training mode
    train_loss, correct = 0.0, 0.0

    progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), disable=not print_batch_stats)
    for batch_idx, (X, y) in progress_bar:        
        X, y = X.to(device), y.to(device)   

        optimizer.zero_grad()
        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        
        loss.backward()
        optimizer.step()  # update the model weights
        optimizer.zero_grad()

        train_loss += loss.item()
        correct += (y_pred.argmax(1)== y).sum().item()

        if print_batch_stats:
            progress_bar.set_description(
                f"Epoch {epoch}/{n_epochs}, "
                f"Batch {batch_idx + 1}/{len(dataloader)}, "
                f"Loss: {loss.item():.6f}"
            )

    # Update the learning rate
    scheduler.step()

    correct /= len(dataloader.dataset)
    return train_loss / len(dataloader), correct

def evaluate(dataloader: DataLoader, model,loss_fn,device,print_batch_stats=True):
        
        model.eval()  # Set the model to evaluation mode
        val_loss, correct = 0.0, 0.0
    
        progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), disable=not print_batch_stats)
    
        with torch.no_grad():
            for batch_idx, (X, y) in progress_bar:        
                X, y = X.to(device), y.to(device)  
                y_pred = model(X)

                loss = loss_fn(y_pred, y)
                    
                val_loss += loss.item()
                correct += (y_pred.argmax(1).float() == y).sum().item()
    
                if print_batch_stats:
                    progress_bar.set_description(f"Batch {batch_idx + 1}/{len(dataloader)}")
    
        correct /= len(dataloader.dataset)
        return val_loss / len(dataloader), correct

N_EPOCHS = 10

X=torch.load('/space/gzanardini/X.pt',weights_only=True)
y=torch.load('/space/gzanardini/Y.pt',weights_only=True).squeeze()
device = "cuda:3" if torch.cuda.is_available() else "cpu"

model=EEGConformer(
    n_outputs=2,
    n_chans=19,
    n_filters_time=50,
    filter_time_length=5,
    pool_time_length=10,
    pool_time_stride=5,
    drop_prob=0.7,
    att_depth=10,
    att_heads=10,
    att_drop_prob=0.3,
    final_fc_length="auto",
    return_features=False,
    n_times=2500,
    chs_info=None,
    add_log_softmax=True
)

model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
loss_fn = nn.CrossEntropyLoss()

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

train_loader = DataLoader(Dataset(x_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(Dataset(x_test, y_test), batch_size=32, shuffle=False)

best_val_loss = float("inf")
best_model = None

train_counts = np.unique(y_train.cpu(), return_counts=True)[1]
test_counts = np.unique(y_test.cpu(), return_counts=True)[1]

print(train_counts)
print(test_counts)

n_epochs = N_EPOCHS

for epoch in range(n_epochs):

    train_loss, train_acc = train_one_epoch(train_loader,model,loss_fn,optimizer,scheduler,epoch,device)
    print(f"Epoch {epoch+1}/{n_epochs}, Train Loss: {train_loss:.6f}, Train Acc: {train_acc:.6f}")

    val_loss, val_acc = evaluate(test_loader,model,loss_fn,device)
    print(f"Epoch {epoch+1}/{n_epochs}, Val Loss: {val_loss:.6f}, Val Acc: {val_acc:.6f}")

    with torch.no_grad():
        preds=model(x_test.to(device))
    confusion_matrix = sklearn.metrics.confusion_matrix(y_test.cpu(), preds.argmax(1).cpu())
    print(confusion_matrix)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model.state_dict()

torch.save(best_model, "EEGConformer_photostim_best.pth")

/users/gzanardini/miniconda3/envs/eeg/lib/python3.12/site-packages/braindecode/models/base.py:180: UserWarning: LogSoftmax final layer will be removed! Please adjust your loss function accordingly (e.g. CrossEntropyLoss)!
  warnings.warn("LogSoftmax final layer will be removed! " +


[553 229]
[119  77]


Epoch 0/10, Batch 25/25, Loss: 1.481361: 100%|██████████| 25/25 [00:03<00:00,  6.35it/s]


Epoch 1/10, Train Loss: 1.598926, Train Acc: 0.560102


Batch 7/7: 100%|██████████| 7/7 [00:00<00:00, 23.37it/s]


...

Epoch 9/10, Val Loss: 0.678008, Val Acc: 0.607143
[[119   0]
 [ 77   0]]


Epoch 9/10, Batch 25/25, Loss: 0.693899: 100%|██████████| 25/25 [00:03<00:00,  6.26it/s]


Epoch 10/10, Train Loss: 0.620456, Train Acc: 0.703325


Batch 7/7: 100%|██████████| 7/7 [00:00<00:00, 22.99it/s]


Epoch 10/10, Val Loss: 0.679832, Val Acc: 0.607143
[[119   0]
 [ 77   0]]
